# A Logic Puzzle

The following exercise is taken from the book 
<a href="https://www.amazon.de/Logeleien-Zweistein-ihren-Antworten-Wegner/dp/B006YF0VUE">"99 Logeleien von Zweistein"</a>.
This book has been published 1968.  It is written by 
<a href="http://de.wikipedia.org/wiki/Thomas_von_Randow">Thomas von Randow</a>.

---
The gentlemen Amann, Bemann, Cemann and Demann are called - not necessarily in the same order - by their first names Erich, Fritz, Gustav and Heiner. They are all married to exactly one woman. We also know the following about them and their wives:

- Either Amann's first name is Heiner, or Bemann's wife is Inge.
- If Cemann is married to Josefa, then - **and only in this case** - Klara's husband is **not** called Fritz.
- If Josefa's husband is **not** called Erich, then Inge is married to Fritz.
- If Luise's husband is called Fritz, then Klara's husband's first name is **not** Gustav.
- If the wife of Fritz is called Inge, then Erich is **not** married to Josefa.
- If Fritz is **not** married to Luise, then Gustav's wife's name is Klara.
- Either Demann is married to Luise, or Cemann is called Gustav.

*What are the full fullnames of these gentlemen, and what are their wives' first names?*

---

We are going to solve this problem by coding it in propositional logic and we will solve the resulting set of clauses using the Davis-Putnam algorithm.  In order to code the problem, we will use the following propositional variables:

- $\texttt{MaleName<}x\texttt{,}z\texttt{>}$ for any male first name $x$ and any surname $z$ expresses
  that the gentleman with first name $x$ has surname $z$.
- $\texttt{Married<}x\texttt{,}y\texttt{>}$ for any male first name $x$ and any female first name $y$ expresses
  that the gentleman with first name $x$ is married to the lady with first name $y$.
- $\texttt{FemaleName<}x\texttt{,}z\texttt{>}$ for any female first name $x$ and any last name $z$ expresses
  that the lady with first name $x$ has surname $z$.

We are using the symbols $\texttt{<}$ and $\texttt{>}$ as part of the propositional variables because we want to show the structure of these variables and the parser for propositional logic accepts these symbols as part of propositional variables.

In [ ]:
import { RecursiveSet as Set, Value, Tuple, flatMap } from 'recursive-set';
import { solve } from './06-Davis-Putnam';

In [ ]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [ ]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

In [ ]:
const FirstMale   = set("Erich",  "Fritz", "Gustav", "Heiner");
const FirstFemale = set("Inge",  "Josefa", "Klara",  "Luise");
const SurNames    = set("Amann", "Bemann", "Cemann", "Demann");

Instead of loading an external string parser, we will define our Propositional Logic types and a few robust helper functions directly in TypeScript. This provides better type-safety and leverages `recursive-set`'s value semantics for Conjunctive Normal Form (CNF) clauses.

In [ ]:
type Variable = string;
type Literal  = Variable | Tuple<['¬', Variable]>;
type Clause   = Set<Literal>;
type Clauses  = Set<Clause>;

function complement(l: Literal): Literal {
    if (typeof l === 'string') { return new Tuple('¬', l); }
    return l.get(1);
}

function not(p: Variable): Literal {
    return new Tuple('¬', p);
}

The function `makeVar(f, x, y)` creates a propositional variable of the form `f<x,y>`.

In [ ]:
function makeVar(f: string, x: string, y: string): Variable {
    return `${f}<${x},${y}>`;
}

In [ ]:
makeVar('Married', 'Heiner', 'Klara')

Given a set of propositional variables $S$, the function `atMostOne(S)` computes a set of clauses expressing the fact that at most one of the variables of $S$ is `True`.

In [ ]:
function atMostOne(S: Set<Variable>): Set<Clause> {
    return S.cartesianProduct(S)
            .filterMap(([p, q]) => p != q,
                       ([p, q]) => set(tpl('¬', p), tpl('¬', q)));
}

Given a set of propositional variables $S$, the function `atLeastOne(S)` computes a set of clauses expressing the fact that at least one of the variables of $S$ is `True`.

In [ ]:
function atLeastOne(S: Set<Variable>): Clauses {
    return set<Clause>(S);
}

$S$ is a set of propositional variables. The expression `exactlyOne(S)` creates a set of clauses.  This set expresses the fact that exactly one of the variables in the set $S$ is true.

In [ ]:
function exactlyOne(S: Set<Variable>): Clauses {
    return atMostOne(S).union(atLeastOne(S));
}

In [ ]:
exactlyOne(set('a', 'b', 'c'))

For two sets $A$ and $B$ that have the same number of elements and a function symbol $f$, the procedure `bijective(A, B)` computes a set of clauses that is equivalent to the formula
$$   \bigl(\forall x \in A: \exists! y \in B: f\langle x, y\rangle\bigr) \wedge
     \bigl(\forall y \in B: \exists! x \in A: f\langle x, y\rangle\bigr)
$$
Here the expression $f\langle x,y\rangle$ is the name of a propositional variable and the expression $\exists!x:p(x)$ is to be read as "There exists exactly one $x$ such that $p(x)$ holds".

In [ ]:
function bijective(A: Set<string>, B: Set<string>, f: string): Clauses {
    let clauses = set<Clause>();
    "your code here"
    return clauses;
}

In [ ]:
bijective(set('a', 'b'), set('x', 'y'), 'f')

We introduce standard logical connectives (`implies`, `exclusiveOr`, `equivalence`, `impliesAnd`) as helper functions that directly yield sets of CNF clauses, bypassing the need for a complex string parser.

In [ ]:
function implies(a: Literal, b: Literal): Clauses {
    // a → b ≡ ¬a ∨ b
    return set(set(complement(a), b));
}

function exclusiveOr(a: Literal, b: Literal): Clauses {
    // a ↔ ¬b ≡ (a ∨ b) ∧ (¬a ∨ ¬b)
    return set(set(a, b), set(complement(a), complement(b)));
}

function equivalence(a: Literal, b: Literal): Clauses {
    // a ↔ b ≡ (¬a ∨ b) ∧ (a ∨ ¬b)
    return set(set(complement(a), b),set(a, complement(b)));
}

function impliesAnd(a: Literal, b: Literal, c: Literal): Clauses {
    // (a ∧ b) → c ≡ ¬a ∨ ¬b ∨ c
    return set(set(complement(a), complement(b), c));
}

In [ ]:
exclusiveOr('p', 'q')

Given a set of male first names `FirstMale`, a set of female first names `FirstFemale`, and a set of surnames `Surnames`,
the function `consistentNames(FirstMale, FirstFemale, Surnames)` returns a set of clauses that ensures that if the man
`x` has the surname `z` and the woman `y` also has the surname `z`, then `x` and `y` have to be married.

In [ ]:
function consistentNames(FirstMale: Set<string>, FirstFemale: Set<string>, SurNames: Set<string>): Clauses {
    let clauses = set<Clause>();
    for (const x of FirstMale) {
        for (const y of FirstFemale) {
            for (const z of SurNames) {
                // MaleName<x,z> ∧ FemaleName<y,z> → Married<x,y>
                const a = makeVar('MaleName', x, z);
                const b = makeVar('FemaleName', y, z);
                const c = makeVar('Married', x, y);
                clauses = clauses.union(impliesAnd(a, b, c));
            }
        }
    }
    return clauses;
}

In [ ]:
consistentNames(FirstMale, FirstFemale, SurNames)

Now we translate the text constraints natively into TS using our typed logic helpers.

In [ ]:
function computeClauses(FirstMale: Set<string>, FirstFemale: Set<string>, SurNames: Set<string>): Clauses {
    let clauses = set<Clause>();
    
    // Jedem männlichen Vornamen ist genau ein Nachname zugeordnet und umgekehrt.
    clauses = "your code here"
    // Jeder Mann ist mit genau einer Frau verheiratet und umgekehrt.
    clauses = "your code here"
    // Jede Frau hat genau einen Nachnamen
    clauses = "your code here"
    // Die Namen sind konsistent.
    clauses = "your code here"
    
    // Entweder ist Amanns Vorname Heiner, oder Bemanns Frau heisst Inge.
    clauses = "your code here"
    
    // Wenn Cemann mit Josefa verheiratet ist, dann – und nur in diesem Falle – heisst Klaras Mann nicht Fritz.
    clauses = "your code here";
    
    // Wenn Josefas Mann nicht Erich heisst, dann ist Inge mit Fritz verheiratet.
    clauses = "your code here"
    
    // Wenn Luises Mann Fritz heisst, dann ist der Vorname von Klaras Mann nicht Gustav.
    clauses = "your code here"
    
    // Wenn die Frau von Fritz Inge heisst, dann ist Erich nicht mit Josefa verheiratet.
    clauses = "your code here"
    
    // Wenn Fritz nicht mit Luise verheiratet ist, dann heisst Gustavs Frau Klara.
    clauses = "your code here"
    
    // Entweder ist Demann mit Luise verheiratet, oder Cemann heisst Gustav.
    clauses = "your code here"
    
    return clauses;
}

In [ ]:
const Clauses = computeClauses(FirstMale, FirstFemale, SurNames);
Clauses

There are 242 different clauses.

In [ ]:
Clauses.size

In [ ]:
function compute_solution(FirstMale: Set<string>, FirstFemale: Set<string>, SurNames: Set<string>): Clauses {
    const systemClauses = computeClauses(FirstMale, FirstFemale, SurNames);
    return solve(systemClauses);
}

In [ ]:
const Solution = compute_solution(FirstMale, FirstFemale, SurNames);

In [ ]:
function arb<T extends Value>(S: Set<T>): T {
    // Using recursive-set's efficient random element access
    return S.pickRandom()!;
}

In [ ]:
function onlyPositive(Solution: Clauses): Set<string> {
    const result = set<string>();
    for (const clause of Solution) {
        const literal = arb(clause);
        if (typeof literal === 'string') {
            result.add(literal);
        }
    }
    return result;
}

In [ ]:
onlyPositive(Solution)

In [ ]:
function extractFirst(s: string): string {
    const m = s.match(/<([A-Za-z]+),/);
    return m ? m[1] : '';
}

function extractSecond(s: string): string {
    const m = s.match(/,([A-Za-z]+)>/);
    return m ? m[1] : '';
}

In [ ]:
function displaySolution(Solution: Clauses) {
    const married: Record<string, string> = {};
    const names: Record<string, string> = {};
    
    for (const unit of Solution) {
        for (const l of unit) {
            if (typeof l === 'string') {
                if (l.startsWith("Married")) {
                    married[extractFirst(l)] = extractSecond(l);
                } else if (l.startsWith("MaleName")) {
                    names[extractFirst(l)] = extractSecond(l);
                }
            }
        }
    }
    for (const x of Object.keys(married)) {
        console.log(`${x} ${names[x]} is married to ${married[x]}.`);
    }
}

In [ ]:
displaySolution(Solution);

## Checking the Uniqueness of the Solution

Given a set of unit clauses $U$, the function `checkUniqueness(U)` returns a clause that is the negation of the set $U$.

In [ ]:
function negateSolution(UnitClauses: Clauses): Set<Literal> {
    const result = set<Literal>();
    for (const unit of UnitClauses) {
        result.add(complement(arb(unit)));
    }
    return result;
}

In [ ]:
negateSolution(set<Clause>(set<Literal>('a'), set<Literal>(tpl('¬', 'b'))))

In [ ]:
function checkUniqueness(Solution: Clauses, baseClauses: Clauses) {
    const negation = negateSolution(Solution);
    const augmentedClauses = baseClauses.union(set<Clause>(negation));
    
    const alternative = solve(augmentedClauses);
    const emptyClause = set<Literal>();
    
    // If alternative contains the empty clause {{}}, it's inconsistent
    if (alternative.has(emptyClause) && alternative.size === 1) {
        console.log("Well done: The solution is unique!");
    } else {
        console.log("ERROR: The solution is not unique!");
    }
}

In [ ]:
checkUniqueness(Solution, Clauses);